# SC-24 : Deploiement sur Testnets

[<< Cross-Chain](SC-23-Cross-Chain.ipynb) | [Mainnet Deploy >>](SC-25-Mainnet-Deploy.ipynb)

---

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Configurer une connexion **Sepolia** via Alchemy/Infura
2. Preparer un wallet et obtenir du **testnet ETH** via faucet
3. **Deployer** un contrat sur un reseau public (Sepolia)
4. **Interagir** avec un contrat deploye sur testnet
5. Utiliser **xrpl-py** pour envoyer des transactions sur le **XRP Testnet**

### Prerequis

- [SC-2-Setup-Web3py](../00-Foundations/SC-2-Setup-Web3py.ipynb) et [SC-3-Solidity-Basics](../01-Solidity-Foundation/SC-3-Solidity-Basics.ipynb)
- Cle API Alchemy ou Infura (gratuit)
- `pip install web3 py-solc-x xrpl-py python-dotenv`

### Duree estimee : 50 minutes

---

## 0. Prerequis

- Cle API **Alchemy** ou **Infura** (gratuit)
- **MetaMask** ou wallet Ethereum pour recevoir du testnet ETH
- Faucet Sepolia : [Google Cloud Faucet](https://cloud.google.com/application/web3/faucet/ethereum/sepolia) ou [Alchemy Faucet](https://sepoliafaucet.com/)
- Python : `pip install web3 py-solc-x xrpl-py python-dotenv`

In [ ]:
# Configuration
import os
from dotenv import load_dotenv

load_dotenv()  # Charge .env si present

# Sepolia via Alchemy (remplacez par votre cle)
SEPOLIA_RPC = os.getenv("SEPOLIA_RPC", "https://eth-sepolia.g.alchemy.com/v2/YOUR_KEY")
PRIVATE_KEY = os.getenv("DEPLOYER_PRIVATE_KEY", "")  # Cle privee du deployer

# Verification
if "YOUR_KEY" in SEPOLIA_RPC:
    print("ATTENTION: Configurez SEPOLIA_RPC dans votre .env")
    print("  1. Creez un compte sur https://www.alchemy.com/")
    print("  2. Creez une app Sepolia")
    print("  3. Copiez l'URL RPC dans .env")
else:
    print(f"RPC configure: {SEPOLIA_RPC[:40]}...")

---

## 1. Connexion a Sepolia

Contrairement a anvil (blockchain locale), Sepolia est un reseau public.
Les transactions prennent ~12 secondes (temps de bloc reel).

In [ ]:
from web3 import Web3
import solcx

# Connexion
w3 = Web3(Web3.HTTPProvider(SEPOLIA_RPC))

if w3.is_connected():
    chain_id = w3.eth.chain_id
    block = w3.eth.block_number
    print(f"Connecte a Sepolia (chain_id={chain_id})")
    print(f"Bloc actuel: {block:,}")
    print(f"Gas price: {w3.from_wei(w3.eth.gas_price, 'gwei'):.2f} gwei")
else:
    print("Impossible de se connecter. Verifiez votre cle API.")

---

## 2. Preparation du deployer

Sur un testnet, vous avez besoin d'un compte avec du testnet ETH.
Utilisez un faucet pour obtenir du Sepolia ETH gratuitement.

In [ ]:
from eth_account import Account

if PRIVATE_KEY:
    account = Account.from_key(PRIVATE_KEY)
    balance = w3.eth.get_balance(account.address)
    print(f"Deployer: {account.address}")
    print(f"Balance: {w3.from_wei(balance, 'ether'):.4f} ETH")
    if balance == 0:
        print("\nBalance nulle! Obtenez du Sepolia ETH via un faucet:")
        print(f"  https://cloud.google.com/application/web3/faucet/ethereum/sepolia")
        print(f"  Adresse a alimenter: {account.address}")
else:
    print("Pas de cle privee configuree.")
    print("Pour generer un nouveau wallet de test:")
    print("  account = Account.create()")
    print("  print(account.address, account.key.hex())")

---

## 3. Deploiement d'un contrat sur Sepolia

Nous deploierons un contrat SimpleStorage - le meme que dans SC-3, mais cette fois sur un vrai reseau public.

In [ ]:
SIMPLE_STORAGE = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract SimpleStorage {
    uint256 public storedValue;
    address public owner;
    
    event ValueChanged(uint256 oldValue, uint256 newValue, address changedBy);
    
    constructor(uint256 _initialValue) {
        storedValue = _initialValue;
        owner = msg.sender;
    }
    
    function set(uint256 _value) external {
        uint256 old = storedValue;
        storedValue = _value;
        emit ValueChanged(old, _value, msg.sender);
    }
    
    function get() external view returns (uint256) {
        return storedValue;
    }
}
"""

# Compiler
SOLC_VERSION = "0.8.28"
installed = [str(v) for v in solcx.get_installed_solc_versions()]
if SOLC_VERSION not in installed:
    solcx.install_solc(SOLC_VERSION)
solcx.set_solc_version(SOLC_VERSION)

compiled = solcx.compile_source(
    SIMPLE_STORAGE, output_values=["abi", "bin"], solc_version=SOLC_VERSION
)
contract_id, contract_interface = compiled.popitem()
print(f"Contrat compile: {contract_id}")
print(f"Bytecode: {len(contract_interface['bin'])} caracteres")
print(f"ABI: {len(contract_interface['abi'])} fonctions/events")

In [ ]:
# Deploiement sur Sepolia (necessite du testnet ETH)
if PRIVATE_KEY and w3.is_connected():
    Contract = w3.eth.contract(
        abi=contract_interface["abi"],
        bytecode=contract_interface["bin"]
    )
    
    # Construire la transaction
    nonce = w3.eth.get_transaction_count(account.address)
    tx = Contract.constructor(42).build_transaction({
        "from": account.address,
        "nonce": nonce,
        "gas": 500000,
        "gasPrice": w3.eth.gas_price,
        "chainId": 11155111,  # Sepolia
    })
    
    # Signer et envoyer
    signed_tx = w3.eth.account.sign_transaction(tx, PRIVATE_KEY)
    tx_hash = w3.eth.send_raw_transaction(signed_tx.raw_transaction)
    print(f"Transaction envoyee: {tx_hash.hex()}")
    print(f"Explorer: https://sepolia.etherscan.io/tx/{tx_hash.hex()}")
    
    # Attendre confirmation (~12s)
    print("Attente de confirmation...")
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=120)
    print(f"Deploye a: {receipt.contractAddress}")
    print(f"Gas utilise: {receipt.gasUsed:,}")
    print(f"Explorer: https://sepolia.etherscan.io/address/{receipt.contractAddress}")
else:
    print("Deploiement non disponible (cle manquante ou pas de connexion)")

---

## 4. Interaction avec le contrat deploye

Une fois deploye, le contrat est permanent et accessible par tous.

In [ ]:
# Interagir avec le contrat sur Sepolia
if PRIVATE_KEY and w3.is_connected() and 'receipt' in dir():
    contract = w3.eth.contract(
        address=receipt.contractAddress,
        abi=contract_interface["abi"]
    )
    
    # Lecture (gratuite, pas de gas)
    value = contract.functions.get().call()
    owner = contract.functions.owner().call()
    print(f"Valeur actuelle: {value}")
    print(f"Owner: {owner}")
    
    # Ecriture (coute du gas)
    tx = contract.functions.set(100).build_transaction({
        "from": account.address,
        "nonce": w3.eth.get_transaction_count(account.address),
        "gas": 100000,
        "gasPrice": w3.eth.gas_price,
        "chainId": 11155111,
    })
    signed = w3.eth.account.sign_transaction(tx, PRIVATE_KEY)
    tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction)
    receipt2 = w3.eth.wait_for_transaction_receipt(tx_hash)
    
    new_value = contract.functions.get().call()
    print(f"Nouvelle valeur: {new_value}")
    print(f"Transaction: https://sepolia.etherscan.io/tx/{tx_hash.hex()}")
else:
    print("Interaction non disponible")

---

## 5. Deploiement XRP Testnet

Le XRP Ledger a son propre testnet avec un faucet integre.

In [ ]:
import xrpl
from xrpl.clients import JsonRpcClient
from xrpl.wallet import generate_faucet_wallet
from xrpl.models.transactions import Payment
from xrpl.transaction import submit_and_wait
from xrpl.utils import xrp_to_drops

# Connexion au testnet XRP
client = JsonRpcClient("https://s.altnet.rippletest.net:51234")

try:
    # Generer un wallet finance par le faucet (gratuit, ~10s)
    print("Generation d'un wallet testnet (faucet)...")
    wallet = generate_faucet_wallet(client, debug=True)
    print(f"Adresse: {wallet.address}")
    print(f"Balance: ~1000 XRP (testnet)")
    
    # Creer un deuxieme wallet pour tester un paiement
    print("\nGeneration d'un second wallet...")
    wallet2 = generate_faucet_wallet(client)
    print(f"Destinataire: {wallet2.address}")
except Exception as e:
    print(f"Erreur: {e}")
    print("Le faucet XRP testnet peut etre temporairement indisponible")

In [ ]:
# Envoyer un paiement XRP sur testnet
try:
    payment = Payment(
        account=wallet.address,
        amount=xrp_to_drops(25),  # 25 XRP
        destination=wallet2.address,
    )
    
    print("Envoi de 25 XRP...")
    response = submit_and_wait(payment, client, wallet)
    result = response.result
    
    print(f"Status: {result['meta']['TransactionResult']}")
    print(f"Hash: {result['hash']}")
    print(f"Explorer: https://testnet.xrpl.org/transactions/{result['hash']}")
    
    # Verifier les balances
    balance1 = xrpl.account.get_balance(wallet.address, client)
    balance2 = xrpl.account.get_balance(wallet2.address, client)
    print(f"\nBalance expediteur: {int(balance1)/1_000_000:.2f} XRP")
    print(f"Balance destinataire: {int(balance2)/1_000_000:.2f} XRP")
except Exception as e:
    print(f"Erreur: {e}")

---

## 6. Comparaison Testnet vs Mainnet

| Aspect | Anvil (local) | Sepolia (testnet) | Mainnet |
|--------|--------------|-------------------|----------|
| **Cout** | Gratuit | Gratuit (faucet) | Reel ($$$) |
| **Vitesse** | Instantane | ~12s (bloc) | ~12s (bloc) |
| **Persistance** | Reset a chaque redemarrage | Permanent | Permanent |
| **Acces** | localhost | Public | Public |
| **Usage** | Dev/debug | Test d'integration | Production |

### Bonnes pratiques
1. **Developper sur anvil** (SC-3 a SC-12)
2. **Tester sur testnet** (ce notebook)
3. **Deployer sur mainnet** seulement quand tout est verifie (SC-25)

---

## Exercice

Deployez un contrat ERC-20 (SC-7) sur Sepolia et effectuez un transfer reel entre deux adresses.
Verifiez le resultat sur Etherscan.

In [ ]:
# Exercice : Deploy ERC-20 sur Sepolia
# 1. Reprendre le contrat SimpleToken de SC-7
# 2. Le compiler avec py-solc-x
# 3. Le deployer sur Sepolia
# 4. Effectuer un transfer()
# 5. Verifier sur https://sepolia.etherscan.io/

raise NotImplementedError("Completez cet exercice")

---

[<< Cross-Chain](SC-23-Cross-Chain.ipynb) | [Mainnet Deploy >>](SC-25-Mainnet-Deploy.ipynb)